In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponibile:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA disponibile: True
GPU: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path

zip_path = Path("/content/drive/MyDrive/deepfake-thesis/deepfake_training.zip")

print("ZIP trovato:", zip_path.exists())

if zip_path.exists():
    print(f"Dimensione: {zip_path.stat().st_size / (1024**3):.2f} GB")

ZIP trovato: True
Dimensione: 4.84 GB


In [4]:
import shutil
from pathlib import Path

local_zip = Path("/content/deepfake_training.zip")

shutil.copy2(zip_path, local_zip)

print("Copia completata:", local_zip.exists())
print(f"Dimensione: {local_zip.stat().st_size / (1024**3):.2f} GB")

Copia completata: True
Dimensione: 4.84 GB


In [5]:
import zipfile
from pathlib import Path

project_dir = Path("/content/deepfake-thesis")
project_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(local_zip, "r") as z:
    z.extractall(project_dir)

print("Estrazione completata")

Estrazione completata


In [6]:
faces_dir = project_dir / "data_processed" / "faces"
metadata_file = project_dir / "metadata" / "dataset_splits.csv"
src_dir = project_dir / "src"

faces = list(faces_dir.glob("*.png"))

print("Crop trovati:", len(faces))
print("dataset_splits.csv:", metadata_file.exists())

print("\nFile in src:")
for file in sorted(src_dir.glob("*.py")):
    print("-", file.name)

Crop trovati: 49653
dataset_splits.csv: True

File in src:
- dataloaders.py
- dataset.py
- models.py
- train_xception.py


In [7]:
!pip install -q timm

In [8]:
import timm
import torch

print("timm:", timm.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Nessuna")

timm: 1.0.29
PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


In [9]:
%cd /content/deepfake-thesis
!python src/train_xception.py --smoke-test

/content/deepfake-thesis
TRAINING XCEPTION BASELINE

Device: cuda
GPU: Tesla T4

MODALITÀ SMOKE TEST ATTIVA
Batch size: 2
Epoche massime: 1

Training samples: 35769
Validation samples: 6990
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to /root/.cache/torch/hub/checkpoints/xception-43020ad28.pth

Distribuzione training set:
REAL (0): 7164
FAKE (1): 28605

Pesi CrossEntropyLoss:
REAL: 2.4964
FAKE: 0.6252

Mixed precision: True

EPOCA 1/1

TRAIN
Loss:      0.8479
Accuracy:  0.0000
Precision: 0.0000
Recall:    0.0000
F1:        0.0000
ROC-AUC:   nan

VALIDATION
Loss:      0.6055
Accuracy:  1.0000
Precision: 0.0000
Recall:    0.0000
F1:        0.0000
ROC-AUC:   nan

Learning rate: 0.00010000
Tempo epoca: 15.1 secondi

Nuova migliore validation loss.

TRAINING TERMINATO

Smoke test completato.


In [10]:
from pathlib import Path
import os
import shutil

project_dir = Path("/content/deepfake-thesis")
drive_outputs = Path("/content/drive/MyDrive/deepfake-thesis/xception_baseline")

drive_models = drive_outputs / "models"
drive_results = drive_outputs / "results"

drive_models.mkdir(parents=True, exist_ok=True)
drive_results.mkdir(parents=True, exist_ok=True)

local_models = project_dir / "models"
local_results = project_dir / "results"

# Eliminiamo le cartelle locali solo se sono vuote
if local_models.exists() and not any(local_models.iterdir()):
    local_models.rmdir()

if local_results.exists() and not any(local_results.iterdir()):
    local_results.rmdir()

# Creiamo collegamenti verso Google Drive
if not local_models.exists():
    os.symlink(drive_models, local_models)

if not local_results.exists():
    os.symlink(drive_results, local_results)

print("models ->", local_models.resolve())
print("results ->", local_results.resolve())

models -> /content/drive/MyDrive/deepfake-thesis/xception_baseline/models
results -> /content/drive/MyDrive/deepfake-thesis/xception_baseline/results


In [11]:
%cd /content/deepfake-thesis

import sys
import torch
import torch.nn as nn

sys.path.insert(0, "/content/deepfake-thesis/src")

from dataloaders import create_dataloaders
from models import create_model
from train_xception import (
    calculate_class_weights,
    train_one_epoch,
    LEARNING_RATE,
    WEIGHT_DECAY,
)

device = torch.device("cuda")

train_loader, _, _ = create_dataloaders(
    model_name="xception",
    batch_size=32,
    num_workers=2,
)

model = create_model(
    model_name="xception",
    pretrained=True,
    num_classes=2,
).to(device)

class_weights = calculate_class_weights(
    train_loader.dataset,
    device
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True
)

torch.cuda.reset_peak_memory_stats()

metrics = train_one_epoch(
    model=model,
    dataloader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    scaler=scaler,
    use_amp=True,
    max_batches=1,
)

print("\nTest batch 32 completato.")
print("Loss:", metrics["loss"])
print(
    f"Picco VRAM: "
    f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)

del model, optimizer, criterion, scaler
torch.cuda.empty_cache()

/content/deepfake-thesis

Distribuzione training set:
REAL (0): 7164
FAKE (1): 28605

Pesi CrossEntropyLoss:
REAL: 2.4964
FAKE: 0.6252

Test batch 32 completato.
Loss: 0.6887677311897278
Picco VRAM: 4.04 GB


In [ ]:
%cd /content/deepfake-thesis
!python -u src/train_xception.py --batch-size 32 --num-workers 2

/content/deepfake-thesis
TRAINING XCEPTION BASELINE

Device: cuda
GPU: Tesla T4
Batch size: 32
Epoche massime: 20

Training samples: 35769
Validation samples: 6990

Distribuzione training set:
REAL (0): 7164
FAKE (1): 28605

Pesi CrossEntropyLoss:
REAL: 2.4964
FAKE: 0.6252

Mixed precision: True

EPOCA 1/20


In [ ]:
%cd /content/deepfake-thesis

import sys
import torch
import torch.nn as nn
import pandas as pd
from pathlib import Path

sys.path.insert(0, "/content/deepfake-thesis/src")

from dataloaders import create_dataloaders
from models import create_model
from train_xception import validate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# 1. DataLoader del test set
# ------------------------------------------------------------

_, _, test_loader = create_dataloaders(
    model_name="xception",
    batch_size=32,
    num_workers=2,
)

print("Campioni test:", len(test_loader.dataset))

# ------------------------------------------------------------
# 2. Caricamento del checkpoint migliore
# ------------------------------------------------------------

checkpoint_path = Path(
    "/content/deepfake-thesis/models/xception_baseline_best.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False,
)

print("Checkpoint epoca:", checkpoint["epoch"])
print(
    "Validation loss checkpoint:",
    checkpoint["validation_loss"]
)

# ------------------------------------------------------------
# 3. Ricreazione di Xception
# ------------------------------------------------------------

model = create_model(
    model_name="xception",
    pretrained=False,
    num_classes=2,
).to(device)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

# ------------------------------------------------------------
# 4. Stessa loss usata durante training/validation
# ------------------------------------------------------------

class_weights = checkpoint["class_weights"].to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# ------------------------------------------------------------
# 5. TEST
# ------------------------------------------------------------

test_metrics = validate(
    model=model,
    dataloader=test_loader,
    criterion=criterion,
    device=device,
    use_amp=(device.type == "cuda"),
)

print("\n" + "=" * 60)
print("RISULTATI TEST XCEPTION BASELINE")
print("=" * 60)

print(f"Loss:      {test_metrics['loss']:.4f}")
print(f"Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall:    {test_metrics['recall']:.4f}")
print(f"F1:        {test_metrics['f1']:.4f}")
print(f"ROC-AUC:   {test_metrics['roc_auc']:.4f}")

# ------------------------------------------------------------
# 6. Salvataggio dei risultati su Drive
# ------------------------------------------------------------

results_path = Path(
    "/content/deepfake-thesis/results/"
    "xception_baseline_test_metrics.csv"
)

pd.DataFrame([test_metrics]).to_csv(
    results_path,
    index=False
)

print("\nRisultati salvati in:")
print(results_path)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history = pd.read_csv(
    "/content/drive/MyDrive/deepfake-thesis/xception_baseline/results/xception_baseline_history.csv"
)

plt.figure(figsize=(8, 5))

plt.plot(
    history["epoch"],
    history["train_loss"],
    marker="o",
    label="Training loss"
)

plt.plot(
    history["epoch"],
    history["val_loss"],
    marker="o",
    label="Validation loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.show()